# Validation template

**Reusable for your own corpus.** Adapt three things and the rest runs unchanged.

1. The CSV path and column names.
2. The hypotheses and weights for your concept.
3. The thresholds that bin scores into &minus;1 / 0 / +1 classes.

Output is a kappa against your gold set, a confusion matrix, and a CSV with paragraph-level scores ready to aggregate.

Runs on free Colab CPU for a few hundred paragraphs. Switch to a paid GPU runtime for larger corpora.

## 1. Setup

In [ ]:
!pip install -q transformers scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import torch
from transformers import pipeline
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from scipy.stats import pearsonr

MODEL_ID = 'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli'
device = 0 if torch.cuda.is_available() else -1
zsc = pipeline('zero-shot-classification', model=MODEL_ID, device=device)
print(f'Loaded {MODEL_ID} on {"GPU" if device == 0 else "CPU"}')

## 2. Load your data

Replace the path. Replace the column names. The template assumes one column with the text and one column with the gold class. The gold column should hold integers in the set {&minus;1, 0, 1}.

In [ ]:
CSV_PATH = '/content/your_corpus.csv'
TEXT_COL = 'text'
GOLD_COL = 'gold'

df = pd.read_csv(CSV_PATH)
df = df[[TEXT_COL, GOLD_COL]].dropna().reset_index(drop=True)
print(f'Loaded {len(df)} rows.')
print(f'Gold class balance: {df[GOLD_COL].value_counts().sort_index().to_dict()}')
df.head(3)

## 3. Define your hypotheses and weights

Each hypothesis is a sentence the model checks for entailment against your text. Positive weights pull the score toward +1. Negative weights pull toward &minus;1. Magnitudes set how much each hypothesis counts.

Start with three hypotheses. Add more after you see where the model misses on your gold set.

In [ ]:
hypotheses = [
    ('The text expresses approval of the target.', +1.0),
    ('The text describes the target in positive terms.', +0.7),
    ('The text criticizes the target.', -1.0),
    ('The text expresses concern about the target.', -0.7),
]
hyp_texts = [h for h, _ in hypotheses]
hyp_weights = np.array([w for _, w in hypotheses])
print(f'{len(hypotheses)} hypotheses defined.')

## 4. Score every paragraph

Each text gets a continuous score in roughly [&minus;1, +1]. The score is the weighted average of entailment probabilities, normalized by the sum of absolute weights.

In [ ]:
def ensemble_scores(texts, hyp_texts, hyp_weights, batch_size=8):
    preds = zsc(texts, candidate_labels=hyp_texts, multi_label=True, batch_size=batch_size)
    weight_norm = np.abs(hyp_weights).sum()
    out = []
    for p in preds:
        prob_by_text = dict(zip(p['labels'], p['scores']))
        probs = np.array([prob_by_text[h] for h in hyp_texts])
        out.append(float((probs * hyp_weights).sum() / weight_norm))
    return out

df['score'] = ensemble_scores(df[TEXT_COL].tolist(), hyp_texts, hyp_weights)
print(f'Score range: {df["score"].min():+.2f} to {df["score"].max():+.2f}')
print(f'Score mean: {df["score"].mean():+.3f}, std: {df["score"].std():.3f}')

## 5. Bin scores into classes

Pick thresholds that split your score distribution into &minus;1 / 0 / +1 in proportions that match your gold class balance. Start with symmetric thresholds. Adjust if your gold is imbalanced.

In [ ]:
HI_THRESHOLD = 0.05
LO_THRESHOLD = -0.05

def to_class(s, hi=HI_THRESHOLD, lo=LO_THRESHOLD):
    if s > hi: return 1
    if s < lo: return -1
    return 0

df['pred'] = df['score'].apply(to_class)
print(f'Predicted class balance: {df["pred"].value_counts().sort_index().to_dict()}')

## 6. Validation against gold

Three numbers go in your appendix.

1. Cohen's kappa. Class agreement, corrected for chance.
2. Pearson correlation between continuous score and gold class. Useful even when gold is integer-coded.
3. Confusion matrix. Tells you which classes the model confuses.

In [ ]:
kappa = cohen_kappa_score(df[GOLD_COL], df['pred'])
r, _ = pearsonr(df['score'], df[GOLD_COL])
acc = (df[GOLD_COL] == df['pred']).mean()

print(f'Cohen kappa     {kappa:+.3f}')
print(f'Pearson r       {r:+.3f}')
print(f'Accuracy        {acc:.3f}  ({(df[GOLD_COL] == df["pred"]).sum()} / {len(df)})')
print()
print('Confusion matrix (rows = gold, cols = predicted):')
print(pd.DataFrame(
    confusion_matrix(df[GOLD_COL], df['pred'], labels=[-1,0,1]),
    index=['gold=-1','gold= 0','gold=+1'],
    columns=['pred=-1','pred= 0','pred=+1'],
))

## 7. Inspect the disagreements

Off-diagonal rows of the confusion matrix. These are the paragraphs to re-read by hand. Look for patterns. If five disagreements share a phrase you did not anticipate, add a hypothesis that covers it.

In [ ]:
misses = df[df[GOLD_COL] != df['pred']].copy()
print(f'{len(misses)} disagreements')
for i, r in misses.head(8).iterrows():
    print(f'\n[row {i}] gold={int(r[GOLD_COL]):+d}  pred={int(r["pred"]):+d}  score={r["score"]:+.2f}')
    text = str(r[TEXT_COL])
    print('  ' + text[:280] + ('...' if len(text) > 280 else ''))

## 8. Save the per-paragraph CSV

What aggregates to a country-year, party-year, or whatever unit your paper uses.

In [ ]:
OUT_PATH = '/content/scored_paragraphs.csv'
df.to_csv(OUT_PATH, index=False)
print(f'Wrote {len(df)} rows to {OUT_PATH}')

## Iteration loop

If your kappa is below 0.6, walk through this list before changing models.

1. Re-read 10 disagreements by hand. Group them by failure mode.
2. For each failure mode, add one hypothesis that targets it. Set its weight by codebook judgment, not by tuning.
3. Re-run the notebook. Note the new kappa.
4. Stop adding hypotheses when kappa stops improving. More than 15 rarely helps and the appendix gets harder to defend.
5. If kappa still will not move, the issue is the codebook, not the model. Re-read your codebook with a co-author.